# Input-Response Space-Filling Design

This notebook constructs a Pareto
front of input-response space-filling designs of 20 runs, over a 2-D input space
(``x1``, ``x2``) with a single response ``y``. Every design on the Pareto front
has a different balance between space-filling in the *input* space and in the
*response* space, letting the experimenter choose the trade-off that best suits
their goal.

The candidate set is a regular 21x21 grid over ``[0, 1]^2`` with a response
generated from a linear model (``y`` ranges from about -4.9 to 20.9).

## 1. Load and inspect the candidate set

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from idaes_sdoe import ColumnRoles, load_csv, prepare_design_setup
from idaes_sdoe.design import design_input_response
from idaes_sdoe.plotting import plot_pair_matrix, plot_pareto_front

candidate = load_csv(Path("supporting_data/irsf-example1-candset.csv"))
print(f"{len(candidate)} candidate points; ranges:")
candidate.describe().loc[["min", "max"]]

The response ``y`` shown in color over the input grid (a contour-style view):

In [ ]:
contour = go.Figure(
    go.Scatter(
        x=candidate["x1"], y=candidate["x2"], mode="markers",
        marker={"color": candidate["y"], "colorscale": "Viridis",
                "colorbar": {"title": "y"}, "size": 9},
    )
)
contour.update_layout(title="Candidate set: response y over the input grid",
                      xaxis_title="x1", yaxis_title="x2", template="plotly_white",
                      width=620, height=520)
contour

The same candidate set with the response shown as point size:

In [ ]:
scaled = (candidate["y"] - candidate["y"].min()) / (candidate["y"].max() - candidate["y"].min())
sizes = go.Figure(
    go.Scatter(
        x=candidate["x1"], y=candidate["x2"], mode="markers",
        marker={"size": 4 + 16 * scaled, "color": "rgba(31,119,180,0.6)"},
    )
)
sizes.update_layout(title="Candidate set: response y shown as point size",
                    xaxis_title="x1", yaxis_title="x2", template="plotly_white",
                    width=620, height=520)
sizes

## 2. Configure roles and run the IRSF search

Input-response space filling only uses the maximin criterion. We mark ``x1`` and
``x2`` as inputs and ``y`` as the response, then build a design of 20 runs.

Here we use a modest number of random starts for a
faster demo; more random starts generally yield a fuller, smoother Pareto front.

In [ ]:
NUM_RESTARTS = 10  # more restarts give a fuller front

setup = prepare_design_setup(
    candidate=candidate,
    roles=ColumnRoles(inputs=["x1", "x2"], responses=["y"]),
)
result = design_input_response(
    setup=setup, design_size=20, num_restarts=NUM_RESTARTS, random_state=1
)
print(f"Pareto-optimal designs found: {result.num_designs}")

## 3. Examine the Pareto front

Each point is one design. Larger values on the x-axis mean better space-filling
in the *input* space; larger values on the y-axis mean better space-filling in
the *response* space. Designs near the ends favor one space heavily; designs
near the middle balance the two.

In [ ]:
result.pareto_front

In [ ]:
plot_pareto_front(result.pareto_front)

## 4. Compare designs across the front

We compare three designs: the **best response
space-filling** design (one end of the front), the **best input space-filling**
design (the other end — equivalent to a plain uniform space-filling design), and
a **compromise** design from the middle. For each, the ``x1``-``x2`` scatter
shows input space-filling and the ``y`` histogram shows response space-filling.

In [ ]:
best_response = 1                       # smallest input criterion, best response filling
best_input = result.num_designs         # largest input criterion, best input filling
compromise = (result.num_designs + 1) // 2
summary = result.pareto_front.set_index("Design").loc[[best_response, compromise, best_input]]
summary

In [ ]:
plot_pair_matrix(result.designs[best_response], columns=["x1", "x2", "y"],
                 candidate=candidate, title=f"Design {best_response}: best response space-filling")

In [ ]:
plot_pair_matrix(result.designs[compromise], columns=["x1", "x2", "y"],
                 candidate=candidate, title=f"Design {compromise}: compromise")

In [ ]:
plot_pair_matrix(result.designs[best_input], columns=["x1", "x2", "y"],
                 candidate=candidate, title=f"Design {best_input}: best input space-filling (uniform)")

The best-response design spreads ``y`` evenly (a full response histogram) but
leaves gaps in the ``x1``-``x2`` scatter; the best-input design covers the input
space uniformly but leaves gaps in ``y``; the compromise design fills both
reasonably. Any design on the Pareto front is optimal for its particular
weighting, so the experimenter picks the balance that fits the study.